# bronze_flight_events

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_bronze.bronze_flight_events;

In [0]:
# dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_flight_events/", True)

False

In [0]:
#dbutils.fs.mv(
 #   "/Volumes/workspace/airline_bronze/volumes/flight_events.csv",
  #  "/Volumes/workspace/airline_bronze/volumes/flight_events/flight_events.csv")

True

In [0]:
#dbutils.fs.ls("/Volumes/workspace/airline_bronze/volumes/flight_events/")

[FileInfo(path='dbfs:/Volumes/workspace/airline_bronze/volumes/flight_events/flight_events.csv', name='flight_events.csv', size=19398, modificationTime=1777890655000)]

In [0]:
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/airline_bronze/volumes/schema/flight_events/") \
    .load("/Volumes/workspace/airline_bronze/volumes/flight_events/")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_flight_events/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_bronze.bronze_flight_events")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_bronze.bronze_flight_events;

COUNT(*)
200


# bronze_baggage_events

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_bronze.bronze_baggage_events;

In [0]:
# dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_baggage_events/", True)

False

In [0]:
# dbutils.fs.mkdirs("/Volumes/workspace/airline_bronze/volumes/baggage_events/")

True

In [0]:
#dbutils.fs.mv(
 #   "/Volumes/workspace/airline_bronze/volumes/baggage_events.csv",
  #  "/Volumes/workspace/airline_bronze/volumes/baggage_events/baggage_events.csv"
#)

True

In [0]:
#dbutils.fs.ls("/Volumes/workspace/airline_bronze/volumes/baggage_events/")

[FileInfo(path='dbfs:/Volumes/workspace/airline_bronze/volumes/baggage_events/baggage_events.csv', name='baggage_events.csv', size=219883, modificationTime=1777891436000)]

In [0]:
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/airline_bronze/volumes/schema/baggage_events/") \
    .load("/Volumes/workspace/airline_bronze/volumes/baggage_events/")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_baggage_events/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_bronze.bronze_baggage_events")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_bronze.bronze_baggage_events;

COUNT(*)
3000


# bronze_gate_operations

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_bronze.bronze_gate_operations;

In [0]:
#dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_gate_operations/", True)

False

In [0]:
#dbutils.fs.mkdirs("/Volumes/workspace/airline_bronze/volumes/gate_operations/")

True

In [0]:
#dbutils.fs.mv(
 #   "/Volumes/workspace/airline_bronze/volumes/gate_operations.csv",
  #  "/Volumes/workspace/airline_bronze/volumes/gate_operations/gate_operations.csv"
#)

True

In [0]:
# dbutils.fs.ls("/Volumes/workspace/airline_bronze/volumes/gate_operations/")

[FileInfo(path='dbfs:/Volumes/workspace/airline_bronze/volumes/gate_operations/gate_operations.csv', name='gate_operations.csv', size=28952, modificationTime=1777891552000)]

In [0]:
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/airline_bronze/volumes/schema/gate_operations/") \
    .load("/Volumes/workspace/airline_bronze/volumes/gate_operations/")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/bronze_gate_operations/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_bronze.bronze_gate_operations")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_bronze.bronze_gate_operations;

COUNT(*)
407


# silver_flight_events

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_silver.silver_flight_events;

In [0]:
#dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_flight_events/", True)

True

In [0]:
from pyspark.sql.functions import col, when, expr

df = spark.readStream.table("workspace.airline_bronze.bronze_flight_events")

df_clean = df \
    .withColumn("scheduled_dep", when(col("scheduled_dep") == "NULL", None).otherwise(col("scheduled_dep"))) \
    .withColumn("actual_dep", when(col("actual_dep") == "NULL", None).otherwise(col("actual_dep"))) \
    .withColumn("event_time", when(col("event_time") == "NULL", None).otherwise(col("event_time")))

df_clean = df_clean \
    .withColumn("scheduled_dep", expr("try_to_timestamp(scheduled_dep)")) \
    .withColumn("actual_dep", expr("try_to_timestamp(actual_dep)")) \
    .withColumn("event_time", expr("try_to_timestamp(event_time)"))

df_clean = df_clean.withColumn(
    "actual_dep",
    when(col("actual_dep").isNull(), col("scheduled_dep"))
    .otherwise(col("actual_dep"))
)

df_clean = df_clean.withColumn(
    "delay_minutes",
    expr("CAST((unix_timestamp(actual_dep) - unix_timestamp(scheduled_dep)) / 60 AS INT)")
)

df_watermarked = df_clean.withWatermark("event_time", "10 minutes")

df_dedup = df_watermarked.dropDuplicates(["flight_id", "event_time"])

df_dedup.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_flight_events/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_silver.silver_flight_events")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_silver.silver_flight_events;

COUNT(*)
0


# silver_baggage_events

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_silver.silver_baggage_events;

In [0]:
#dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_baggage_events/", True)

False

In [0]:
from pyspark.sql.functions import col, when, expr, broadcast


df = spark.readStream.table("workspace.airline_bronze.bronze_baggage_events")


df_clean = df.filter(col("bag_tag_id").isNotNull())


df_clean = df_clean \
    .withColumn("event_time", when(col("event_time") == "NULL", None).otherwise(col("event_time"))) \
    .withColumn("belt_number", when(col("belt_number") == "NULL", None).otherwise(col("belt_number"))) \
    .withColumn("is_mishandled", when(col("is_mishandled") == "NULL", None).otherwise(col("is_mishandled")))

df_clean = df_clean \
    .withColumn("bag_tag_id", col("bag_tag_id").cast("string")) \
    .withColumn("flight_id", col("flight_id").cast("string")) \
    .withColumn("passenger_id", col("passenger_id").cast("string")) \
    .withColumn("origin_airport", col("origin_airport").cast("string")) \
    .withColumn("destination_airport", col("destination_airport").cast("string")) \
    .withColumn("status", col("status").cast("string")) \
    .withColumn("belt_number", col("belt_number").cast("string")) \
    .withColumn("is_mishandled", col("is_mishandled").cast("boolean")) \
    .withColumn("event_time", expr("try_to_timestamp(event_time)"))


df_clean = df_clean.withColumn(
    "belt_number",
    when(col("belt_number").isNull(), "UNKNOWN").otherwise(col("belt_number"))
)


df_flights = spark.read.table("workspace.airline_silver.silver_flight_events") \
    .select("flight_id").distinct()

df_valid = df_clean.join(
    broadcast(df_flights),
    on="flight_id",
    how="inner"
)

df_watermarked = df_valid.withWatermark("event_time", "10 minutes")


df_dedup = df_watermarked.dropDuplicates(["bag_tag_id", "event_time"])


df_dedup.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_baggage_events/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_silver.silver_baggage_events")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_silver.silver_flight_events;

COUNT(*)
200


# silver_gate_operations

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.airline_silver.silver_gate_operations;

In [0]:
# dbutils.fs.rm("/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_gate_operations/", True)

True

In [0]:
from pyspark.sql.functions import col, when, expr, upper


df = spark.readStream.table("workspace.airline_bronze.bronze_gate_operations")

df_clean = df \
    .withColumn("event_time", when(col("event_time") == "NULL", None).otherwise(col("event_time"))) \
    .withColumn("previous_gate", when(col("previous_gate") == "NULL", None).otherwise(col("previous_gate"))) \
    .withColumn("conflict_flight", when(col("conflict_flight") == "NULL", None).otherwise(col("conflict_flight")))


df_clean = df_clean \
    .withColumn("gate_event_id", upper(col("gate_event_id").cast("string"))) \
    .withColumn("flight_id", upper(col("flight_id").cast("string"))) \
    .withColumn("gate_number", upper(col("gate_number").cast("string"))) \
    .withColumn("terminal", upper(col("terminal").cast("string"))) \
    .withColumn("event_type", upper(col("event_type").cast("string"))) \
    .withColumn("previous_gate", upper(col("previous_gate").cast("string"))) \
    .withColumn("conflict_flight", upper(col("conflict_flight").cast("string")))


df_clean = df_clean.withColumn(
    "event_time",
    expr("try_to_timestamp(event_time)")
)

df_clean = df_clean.withColumn(
    "is_conflict",
    when(col("event_type") == "CONFLICT DETECTED", True).otherwise(False)
)


df_clean = df_clean.filter(col("gate_event_id").isNotNull())


df_watermarked = df_clean.withWatermark("event_time", "10 minutes")


df_dedup = df_watermarked.dropDuplicates(["gate_event_id", "event_time"])


df_dedup.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/airline_bronze/volumes/checkpoints/silver_gate_operations_new/") \
    .trigger(availableNow=True) \
    .toTable("workspace.airline_silver.silver_gate_operations")

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.airline_silver.silver_gate_operations;

COUNT(*)
407
